# WP3b: SAR Ice Classification — Hornsund Method

**Owner: Julian**

Implements the GLCM + SVM pipeline from Williams & Swirad (2025), adapted for Sermilik Fjord.

**Pipeline:**
1. Build 10-band GLCM composite (same as `02b`)
2. Load training polygons (from colleague's `03a`) → sample pixels → 40/60 split
3. Train SVM (RBF kernel)
4. Evaluate on test set → confusion matrix
5. Classify full collection → multi-class `ice_type_s1`
6. Collapse to binary `ice_s1` for time series fusion
7. Compute and export ice area time series

**Ice classes:**

| Value | Class |
|---|---|
| 0 | Open water |
| 1 | Drift ice |
| 2 | Landfast ice |
| 3 | Glacier ice |

> **Dependency:** set `TRAINING_ASSET` below once your colleague uploads training polygons from `03a`.

In [ ]:
import ee
import geemap
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project, export_to_drive
from src.preprocessing_s1 import filter_dual_pol, preprocess_s1, COMPOSITE_BANDS
from src.classification_s1 import train_svm, classify_svm, to_binary
from src.timeseries import compute_ice_area, collection_to_timeseries, detect_anomalies

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

# --- Set your training asset path here once available ---
TRAINING_ASSET = None  # e.g. 'projects/<your-project>/assets/sermilik_training_polygons'

## 3b.1 Build S1 GLCM composite collection

Identical pipeline to `02b`. GEE is lazy — no computation happens here.

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['HH', 'HV'])
)
s1 = filter_dual_pol(s1).map(preprocess_s1)
print('S1 images:', s1.size().getInfo())

## 3b.2 Load S2 reference imagery for training guidance

S2 true-colour and NDSI are used as a visual reference when digitising training polygons.
Pick scenes close in time to winter and summer S1 acquisitions.

In [ ]:
from src.preprocessing_s2 import preprocess_s2

s2_ref = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(preprocess_s2)
)

# Median composite per season — .first() only returns one tile (~110 km swath)
# which leaves the upper fjord uncovered. .median() mosaics all tiles in the window.
s2_winter = s2_ref.filterDate('2021-01-01', '2021-04-30').median().clip(aoi)
s2_summer = s2_ref.filterDate('2021-06-01', '2021-09-30').median().clip(aoi)

# Mean S1 composite for the same windows
s1_winter = s1.filterDate('2021-01-01', '2021-04-30').mean().clip(aoi)
s1_summer = s1.filterDate('2021-06-01', '2021-09-30').mean().clip(aoi)

Map_ref = geemap.Map()
Map_ref.centerObject(aoi, zoom=9)
Map_ref.addLayer(s2_winter, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 True Colour — Winter')
Map_ref.addLayer(s2_winter.select('NDSI'), {'min': -0.5, 'max': 1, 'palette': ['#1a6faf', 'white']}, 'NDSI — Winter', shown=False)
Map_ref.addLayer(s1_winter.select('HH'), {'min': -25, 'max': 0, 'palette': ['black', 'white']}, 'S1 HH — Winter', shown=False)
Map_ref.addLayer(s2_summer, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 True Colour — Summer', shown=False)
Map_ref.addLayer(s1_summer.select('HH'), {'min': -25, 'max': 0, 'palette': ['black', 'white']}, 'S1 HH — Summer', shown=False)
Map_ref

## 3b.3 Training data

Load the training FeatureCollection from `03a` and sample the GLCM composite.

**How to create training polygons (if not done yet):**
1. In GEE Code Editor, use the map above as a visual guide
2. Digitise polygons for each class across both winter and summer scenes
3. Add a `class` integer property (0–3) to each polygon
4. Export as a GEE FeatureCollection asset
5. Set `TRAINING_ASSET` at the top of this notebook

**Recommended coverage:** ≥ 10 polygons per class per season, distributed across the fjord.

In [ ]:
if TRAINING_ASSET is None:
    print('Set TRAINING_ASSET at the top of the notebook to continue.')
else:
    training_fc = ee.FeatureCollection(TRAINING_ASSET)
    print('Training polygons:', training_fc.size().getInfo())
    print('Class distribution:', training_fc.aggregate_histogram('class').getInfo())

In [ ]:
if TRAINING_ASSET is not None:
    # Use the median composite over winter months as the training image
    # (median reduces noise from individual scenes)
    training_image = (
        s1.filterDate('2020-11-01', '2021-04-30')
        .median()
        .select(COMPOSITE_BANDS)
    )

    # Sample pixels inside each training polygon
    samples = training_image.sampleRegions(
        collection=training_fc,
        properties=['class'],
        scale=50,
        tileScale=4,
        geometries=False,
    )

    # 40 / 60 train / test split using a random column
    samples = samples.randomColumn('random', seed=42)
    train_samples = samples.filter(ee.Filter.lt('random', 0.4))
    test_samples  = samples.filter(ee.Filter.gte('random', 0.4))

    print('Train samples:', train_samples.size().getInfo())
    print('Test samples: ', test_samples.size().getInfo())

## 3b.4 Train SVM

RBF kernel SVM via GEE's `libsvm` implementation.
Default hyperparameters: `gamma=0.5`, `cost=10` (from Williams & Swirad 2025).
Tune these once you have training data and can run cross-validation.

In [ ]:
if TRAINING_ASSET is not None:
    classifier = train_svm(train_samples)

    # Training accuracy (optimistic — will be higher than test accuracy)
    train_acc = train_samples.classify(classifier).errorMatrix('class', 'classification')
    print('Training overall accuracy:', train_acc.accuracy().getInfo())
    print('Training kappa:           ', train_acc.kappa().getInfo())

## 3b.5 Accuracy assessment on test set

Uses the held-out 60 % of samples. Report overall accuracy, per-class producer/consumer
accuracy, and Cohen's kappa.

In [ ]:
if TRAINING_ASSET is not None:
    test_classified = test_samples.classify(classifier)
    confusion = test_classified.errorMatrix('class', 'classification')

    print('Test overall accuracy:', confusion.accuracy().getInfo())
    print('Test kappa:           ', confusion.kappa().getInfo())

    print('\nConfusion matrix (rows = actual, cols = predicted):')
    cm = confusion.getInfo()
    class_names = ['Water', 'Drift', 'Landfast', 'Glacier']
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    print(df_cm.to_string())

    print('\nPer-class producer accuracy:', confusion.producersAccuracy().getInfo())
    print('Per-class consumer accuracy:', confusion.consumersAccuracy().getInfo())

## 3b.6 Classify full collection

Applies the trained SVM to every image in the collection.
Each image gets two new bands:
- `ice_type_s1`: multi-class label (0–3)
- `ice_s1`: binary ice / water (1 / 0)

In [ ]:
if TRAINING_ASSET is not None:
    s1_classified = s1.map(
        lambda img: to_binary(classify_svm(img, classifier))
    )
    print('Classified collection size:', s1_classified.size().getInfo())

## 3b.7 Visualise classification result

In [ ]:
if TRAINING_ASSET is not None:
    sample_classified = (
        s1_classified
        .filterDate('2021-01-01', '2021-03-31')
        .first()
        .clip(aoi)
    )

    # 0=water  1=drift  2=landfast  3=glacier
    ice_type_palette = ['#1a6faf', '#a8d8ea', '#ffffff', '#b0c4de']

    Map3 = geemap.Map()
    Map3.centerObject(aoi, zoom=9)
    Map3.addLayer(
        sample_classified.select('ice_type_s1'),
        {'min': 0, 'max': 3, 'palette': ice_type_palette},
        'Ice type (SVM)'
    )
    Map3.addLayer(
        sample_classified.select('ice_s1'),
        {'min': 0, 'max': 1, 'palette': ['#1a6faf', 'white']},
        'Ice binary',
        shown=False
    )
    Map3.addLayer(
        sample_classified.select('HH'),
        {'min': -25, 'max': 0, 'palette': ['black', 'white']},
        'HH backscatter',
        shown=False
    )
    Map3

## 3b.8 Ice area time series

Compute total ice-covered area (km²) per image using the binary `ice_s1` band.
Fetches all image dates from GEE — **this cell is slow** (~minutes for 6 years).

In [ ]:
if TRAINING_ASSET is not None:
    print('Computing ice area time series... (this may take a few minutes)')
    df = collection_to_timeseries(s1_classified, aoi, band='ice_s1')
    df = detect_anomalies(df, column='ice_area_km2')
    print(f'Time series: {len(df)} observations from {df.date.min().date()} to {df.date.max().date()}')
    print(df.head())

In [ ]:
if TRAINING_ASSET is not None:
    fig, ax = plt.subplots(figsize=(14, 4))

    ax.plot(df.date, df.ice_area_km2, color='#4575b4', linewidth=0.8, alpha=0.6, label='Ice area (S1 SVM)')
    ax.plot(df.date, df.rolling_mean,  color='#d73027', linewidth=1.5, label='30-day rolling mean')
    ax.scatter(
        df.loc[df.anomaly, 'date'],
        df.loc[df.anomaly, 'ice_area_km2'],
        color='orange', s=20, zorder=5, label='Anomaly (>2σ)'
    )

    ax.set_xlabel('Date')
    ax.set_ylabel('Ice area (km²)')
    ax.set_title('Sermilik Fjord — S1 SVM ice area 2019–2024')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/figures/s1_svm_timeseries.png', dpi=150)
    plt.show()

## 3b.9 Export

Export the time series to CSV and optionally the classified collection to Google Drive.

In [ ]:
if TRAINING_ASSET is not None:
    out_path = '../outputs/csv/s1_svm_ice_area.csv'
    df[['date', 'ice_area_km2', 'rolling_mean', 'anomaly']].to_csv(out_path, index=False)
    print('Saved:', out_path)

In [ ]:
# Export a single classified image to Google Drive for visual inspection in QGIS.
# Uncomment to run.
#
# if TRAINING_ASSET is not None:
#     export_img = s1_classified.filterDate('2021-01-01', '2021-03-31').first()
#     export_to_drive(
#         image=export_img.select(['ice_type_s1', 'ice_s1', 'HH', 'HV']),
#         description='s1_svm_classified_202101',
#         aoi=aoi,
#         scale=50,
#     )
print('Drive export: uncomment when ready.')

## Notes

- **Hyperparameter tuning:** once training data is ready, test `gamma` in [0.1, 0.5, 1.0]
  and `cost` in [1, 10, 100] using the test-set kappa as the selection criterion.
- **Geometric reclassification** (object orientation/roundness per segment from Williams &
  Swirad 2025) is not implemented here — GEE cannot compute per-object principal-axis
  orientation. This step is deferred to a Python post-processing notebook using
  `rasterio` + `shapely`.
- **Fusion with S2** happens in `04_timeseries.ipynb` using `ice_s1` and `ice_s2` bands.